In [11]:
import pandas as pd

# dataframe = pd.read_csv("./data/ecommerce_data.csv", index_col=0)
dataframe = pd.read_csv("./data/ecommerce_data.csv")
dataframe.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [12]:
# dataframe["InvoiceDate"] = pd.to_datetime(dataframe["InvoiceDate"], dayfirst=True)
dataframe["InvoiceDate"] = pd.to_datetime(dataframe["InvoiceDate"])
# dataframe["InvoiceNo"] = dataframe["InvoiceNo"].astype(int)
# dataframe["CustomerID"] = dataframe["CustomerID"].astype(int)
dataframe.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [13]:
len(dataframe)

541909

In [14]:
import sqlite3

# DB setup for retrieval
connection = sqlite3.connect("database.db")
dataframe.to_sql("ecommerce_table", connection, if_exists="replace")


541909

In [27]:
from google.genai import Client, types
from dotenv import load_dotenv
import os
load_dotenv()

gemini = Client(api_key=os.getenv("GEMINI_API_KEY"))

# SYSTEM_PROMPT = """"""
# gemini_config = types.GenerateContentConfig(
#     system_instruction=SYSTEM_PROMPT,
#     thinking_config=types.ThinkingConfig(
#         thinking_budget=0
#     )
# )

result = gemini.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Write a haiku about the glory to achieved from intelligence, passionate work, wisdom and character"
)

print(result.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [29]:
def sql_tool(query: str):
    """Run a SQL SELECT query on a SQLite database and return the results."""
    return pd.read_sql_query(query, connection).to_dict(orient="records")

In [17]:
sql_tool(
    "SELECT * FROM ecommerce_table;"
)

[{'index': 0,
  'InvoiceNo': '536365',
  'StockCode': '85123A',
  'Description': 'WHITE HANGING HEART T-LIGHT HOLDER',
  'Quantity': 6,
  'InvoiceDate': '2010-12-01 08:26:00',
  'UnitPrice': 2.55,
  'CustomerID': 17850.0,
  'Country': 'United Kingdom'},
 {'index': 1,
  'InvoiceNo': '536365',
  'StockCode': '71053',
  'Description': 'WHITE METAL LANTERN',
  'Quantity': 6,
  'InvoiceDate': '2010-12-01 08:26:00',
  'UnitPrice': 3.39,
  'CustomerID': 17850.0,
  'Country': 'United Kingdom'},
 {'index': 2,
  'InvoiceNo': '536365',
  'StockCode': '84406B',
  'Description': 'CREAM CUPID HEARTS COAT HANGER',
  'Quantity': 8,
  'InvoiceDate': '2010-12-01 08:26:00',
  'UnitPrice': 2.75,
  'CustomerID': 17850.0,
  'Country': 'United Kingdom'},
 {'index': 3,
  'InvoiceNo': '536365',
  'StockCode': '84029G',
  'Description': 'KNITTED UNION FLAG HOT WATER BOTTLE',
  'Quantity': 6,
  'InvoiceDate': '2010-12-01 08:26:00',
  'UnitPrice': 3.39,
  'CustomerID': 17850.0,
  'Country': 'United Kingdom'},
 {'

In [44]:
SYSTEM_PROMPT = """
You are an expert SQL analyst. When appropriate, generate SQL queries based on the user's question and the database schema.
When you generate a query, user 'sql_tool' function to execute the query on the database and get the results.
Then use the results to answer the user's question.

database_schema: [
    {
        table: 'ecommerce_table',
        columns: [
            {
                name: 'InvoiceNo',
                type: 'string'
            },
            {
                name: 'StockCode',
                type: 'string'
            },
            {
                name: 'Description',
                type: 'string'
            },
            {
                name: 'Quantity',
                type: 'int'
            },
            {
                name: 'InvoiceDate',
                type: 'datetime'
            },
            {
                name: 'UnitPrice',
                type: 'float'
            },
            {
                name: 'CustomerID',
                type: 'float'
            },
            {
                name: 'Country',
                type: 'string'
            }
        ]
    }
]
""".strip()

In [50]:
gemini_config_with_tools = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    tools=[sql_tool]
)

In [51]:
chat_session = gemini.chats.create(
    model="gemini-2.5-flash",
    config=gemini_config_with_tools
)

In [ ]:
chat_session.send_message(
    # config=gemini_config_with_tools,
    message="Which item has the highest quantity from United Kingdom?"
)

GenerateContentResponse(
  automatic_function_calling_history=[
    UserContent(
      parts=[
        Part(
          text='Which item has the highest quantity from United Kingdom?'
        ),
      ],
      role='user'
    ),
    Content(
      parts=[
        Part(
          function_call=FunctionCall(
            args={<... 1 item at Max depth ...>},
            name='sql_tool'
          )
        ),
      ],
      role='model'
    ),
    Content(
      parts=[
        Part(
          function_response=FunctionResponse(
            name='sql_tool',
            response={<... 1 item at Max depth ...>}
          )
        ),
      ],
      role='user'
    ),
    Content(
      parts=[
        Part(
          text='The item with the highest quantity from the United Kingdom is "WORLD WAR 2 GLIDERS ASSTD DESIGNS" with a total quantity of 48,326.'
        ),
      ],
      role='model'
    ),
    UserContent(
      parts=[
        Part(
          text='Which item has the highest quantity

In [49]:
chat_session.send_message("What have we discussed till now?")

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='We have discussed and identified the item with the highest quantity from the United Kingdom, which is "WORLD WAR 2 GLIDERS ASSTD DESIGNS" with a total quantity of 48,326.'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='bT-1abKvMoDCjuMP58zzyAU',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=45,
    prompt_token_count=587,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=587
      ),
    ],
    total_token_count=632
  )
)

In [52]:
gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents="What item has the highest quantity from United Kingdom?",
    config=gemini_config_with_tools
)

GenerateContentResponse(
  automatic_function_calling_history=[
    UserContent(
      parts=[
        Part(
          text='What item has the highest quantity from United Kingdom?'
        ),
      ],
      role='user'
    ),
    Content(
      parts=[
        Part(
          function_call=FunctionCall(
            args={<... 1 item at Max depth ...>},
            name='sql_tool'
          )
        ),
      ],
      role='model'
    ),
    Content(
      parts=[
        Part(
          function_response=FunctionResponse(
            name='sql_tool',
            response={<... 1 item at Max depth ...>}
          )
        ),
      ],
      role='user'
    ),
  ],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='The item with the highest quantity from the United Kingdom is "WORLD WAR 2 GLIDERS ASSTD DESIGNS" with a total quantity of 48,326.'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP

In [4]:
import fitz

# PDF DATA

policy_file_path = "./data/ecommerce_policies.pdf"
policy_doc = fitz.open(policy_file_path)

for i, page in enumerate(policy_doc):
    text = page.get_text()
    print(f"Page {i}:\n\n{text}\n\n")

Page 0:

E-COMMERCE POLICIES  
Complete Terms & Conditions  
Effective Date: February 04, 2026  
Version 1.0 
TABLE OF CONTENTS  
1. Terms and Conditions 3 2. Privacy Policy 5 3. Shipping Policy 7 4. 
Return and Refund Policy 8 5. Cookie Policy 9 6. Acceptable Use Policy 10 
7. Intellectual Property Rights 11 8. Disclaimer and Limitation of Liability 12 
1. TERMS AND CONDITIONS  
1.1 Introduction  
Welcome to our e-commerce platform. These Terms and Conditions govern your use of our website 
and the purchase of products through our online store. By accessing or using our website, you agree to 
be bound by these terms. If you do not agree with any part of these terms, you must not use our 
website.  
1.2 Definitions  
In these Terms and Conditions: 'Website' refers to our e-commerce platform and all associated 
services; 'User', 'You', and 'Your' refers to the person accessing this website; 'We', 'Us', and 'Our' refers 
to the company operating this e-commerce platform; 'Products' refer

In [5]:
import faiss
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-large-instruct")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7039.90it/s]


In [6]:
p=0
content_chunks = []
for page in policy_doc:
    page_text = page.get_text("blocks")
    print(f"page {p}:\n")
    b=0
    for block in page_text:
        print(f"block {b}:\n{block}")
        b+=1
        if block[6] == 0: # Only text
            content_chunks.append(block[4])
    print("\n")
    p+=1

page 0:

block 0:
(163.249267578125, 79.31343841552734, 461.9408264160156, 106.12593841552734, 'E-COMMERCE POLICIES  \n', 0, 0)
block 1:
(78.5999984741211, 164.85818481445312, 285.9110412597656, 180.49880981445312, 'Complete Terms & Conditions  \n', 1, 0)
block 2:
(78.5999984741211, 224.95925903320312, 233.44467163085938, 236.13113403320312, 'Effective Date: February 04, 2026  \n', 2, 0)
block 3:
(77.8499984741211, 251.52725219726562, 257.9502258300781, 282.0006103515625, 'Version 1.0 \nTABLE OF CONTENTS  \n', 3, 0)
block 4:
(148.35000610351562, 312.0252990722656, 482.63482666015625, 323.1971740722656, '1. Terms and Conditions 3 2. Privacy Policy 5 3. Shipping Policy 7 4. \n', 4, 0)
block 5:
(147.60000610351562, 332.7972717285156, 482.65283203125, 343.9691467285156, 'Return and Refund Policy 8 5. Cookie Policy 9 6. Acceptable Use Policy 10 \n', 5, 0)
block 6:
(147.60000610351562, 353.7972717285156, 480.783203125, 364.9691467285156, '7. Intellectual Property Rights 11 8. Disclaimer and 

In [7]:
len(content_chunks)

105

In [55]:
# content_embeddings = embedding_model.encode(content_chunks[:10], show_progress_bar=True, batch_size=32)
content_embeddings = embedding_model.encode(content_chunks, show_progress_bar=True, batch_size=32)

Batches: 100%|██████████| 4/4 [05:38<00:00, 84.62s/it] 


In [56]:
vec_dim = content_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(vec_dim)
faiss_index.add(content_embeddings)

In [57]:
query = embedding_model.encode(["How to pay for things?"])

In [58]:
D, I = faiss_index.search(query, 10)
print(I)

[[21 39 23 22 65 59 61 15 30 44]]


In [59]:
print(D)

[[0.25484687 0.3494315  0.35030007 0.35319036 0.36422327 0.37134516
  0.37187517 0.3727543  0.37684286 0.37785578]]


In [60]:
[f"{i}: {content_chunks[i]}" for i in I[0]]

['21: 1.6 Payment  \n',
 '39: 2.5 Your Rights and Choices  \n',
 '23: debit cards, PayPal, and other electronic payment methods. Payment must be received in full before \nyour order is processed. All payments are processed securely through encrypted connections. You \nrepresent and warrant that you have the legal right to use any payment method you provide to us.  \n',
 '22: We accept various payment methods as indicated on our website, which may include credit cards, \n',
 '65: 5.1 What Are Cookies  \n',
 '59: 4.3 Refund Processing  \n',
 '61: 4.4 Exchanges  \n',
 '15: 1.4 Product Information and Pricing  \n',
 '30: 2.2 How We Use Your Information  \n',
 '44: 3.1 Shipping Methods and Costs  \n']

In [61]:
import pickle

faiss.write_index(faiss_index, "policy_index.faiss")
with open("policy_metadata.pkl", "wb") as f:
    pickle.dump(content_chunks, f)

In [62]:
def policy_query_tool(query: str):
    """Fetch relevant information chunks from policy document"""
    try:
        policy_index = faiss.read_index("./policy_index.faiss")
        with open("./policy_metadata.pkl", "rb") as f:
            content_chunks = pickle.load(f)
        
        query_embedding = embedding_model.encode([query])
        D, I = policy_index.search(query_embedding, 10)
        return [f"{i}: {content_chunks[i]}" for i in I[0]]
    except Exception as e:
        print(f"Error in running policy query: {str(e)}")
        return e

In [63]:
SYSTEM_PROMPT = """
You are a customer support agent.
Your task is to fetch relevant information as per the user's query, if possible, from either or both: policy document or ecommerce database.
You have 2 tools available:
1. sql_tool - to fetch information from the ecommerce database
2. policy_query_tool - to fetch relevant information from the policy document

## SQL tool
You have to act as an expert SQL analyst. When appropriate, generate SQL queries based on the user's question and the database schema.
When you generate a query, user 'sql_tool' function to execute the query on the database and get the results.
Then use the results to answer the user's question.

database_schema: [
    {
        table: 'ecommerce_table',
        columns: [
            {
                name: 'InvoiceNo',
                type: 'string'
            },
            {
                name: 'StockCode',
                type: 'string'
            },
            {
                name: 'Description',
                type: 'string'
            },
            {
                name: 'Quantity',
                type: 'int'
            },
            {
                name: 'InvoiceDate',
                type: 'datetime'
            },
            {
                name: 'UnitPrice',
                type: 'float'
            },
            {
                name: 'CustomerID',
                type: 'float'
            },
            {
                name: 'Country',
                type: 'string'
            }
        ]
    }
]
""".strip()

gemini_config_with_tools = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    tools=[sql_tool, policy_query_tool]
)

chat_session = gemini.chats.create(
    model="gemini-2.5-flash",
    config=gemini_config_with_tools
)


In [64]:
response = chat_session.send_message("What are the payment methods that i can use?")
response.automatic_function_calling_history

[UserContent(
   parts=[
     Part(
       text='What are the payment methods that i can use?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'payment methods'
         },
         name='policy_query_tool'
       )
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='policy_query_tool',
         response={
           'result': [
             """21: 1.6 Payment  
 """,
             """59: 4.3 Refund Processing  
 """,
             """0: E-COMMERCE POLICIES  
 """,
             """61: 4.4 Exchanges  
 """,
             """15: 1.4 Product Information and Pricing  
 """,
             <... 5 more items ...>,
           ]
         }
       )
     ),
   ],
   role='user'
 )]

In [65]:
response.automatic_function_calling_history[2].parts[0].function_response.response

{'result': ['21: 1.6 Payment  \n',
  '59: 4.3 Refund Processing  \n',
  '0: E-COMMERCE POLICIES  \n',
  '61: 4.4 Exchanges  \n',
  '15: 1.4 Product Information and Pricing  \n',
  '22: We accept various payment methods as indicated on our website, which may include credit cards, \n',
  '18: 1.5 Order Acceptance and Confirmation  \n',
  '44: 3.1 Shipping Methods and Costs  \n',
  '12: 1.3 Account Registration  \n',
  '103: 8.6 Contact Information  \n']}

In [66]:
response.text

'We accept various payment methods as indicated on our website, which may include credit cards.'

In [4]:
import whisper

stt_model = whisper.load_model("tiny")

In [6]:
# transcription = stt_model.transcribe("audio.mp3")
transcription = stt_model.transcribe("./harvard.wav")

/home/rovince/miniconda3/envs/voicechat/lib/python3.13/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [7]:
print(transcription["text"])

 The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health in zest. A salt pickle tastes fine with ham. Tuckles all pastora are my favorite. A zestful food is the hot cross bun.
